In [15]:
import pandas as pd
import numpy as np
import os


In [16]:
print("DATA PREPROCESSING - CONSUMER COMPLAINTS")
print("\nSTEP 1: Loading CSV file...")

csv_path = '/Users/sristiprasad/Documents/GitHub/customer_complaint_classifier/data/raw_data/consumer_complaints.csv'

try:
    df = pd.read_csv(csv_path)
    print(f"✓ Successfully loaded!")
    print(f"  Shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ File not found: {csv_path}")
    print("Please check the file path!")
    exit()

DATA PREPROCESSING - CONSUMER COMPLAINTS

STEP 1: Loading CSV file...


/var/folders/yb/j_qwq6hs4xz24dxkl79r1p_m0000gn/T/ipykernel_45225/3224142329.py:7: DtypeWarning: Columns (4,5,6,11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


✓ Successfully loaded!
  Shape: (1282355, 18)


In [17]:

print("STEP 2: Exploring data structure...")


print(f"\nTotal records: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

print("\nData types:")
print(df.dtypes)

print("\nFirst 3 rows:")
print(df.head(3))

STEP 2: Exploring data structure...

Total records: 1282355
Total columns: 18

Column names:
  1. Date received
  2. Product
  3. Sub-product
  4. Issue
  5. Sub-issue
  6. Consumer complaint narrative
  7. Company public response
  8. Company
  9. State
  10. ZIP code
  11. Tags
  12. Consumer consent provided?
  13. Submitted via
  14. Date sent to company
  15. Company response to consumer
  16. Timely response?
  17. Consumer disputed?
  18. Complaint ID

Data types:
Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Consumer consent provided?      object
Submitted via                   object
Date sent to comp

In [18]:

print("STEP 3: Identifying text and category columns...")


# Find text column
text_keywords = ['complaint', 'narrative', 'description', 'text', 'message', 'issue']
text_col = None

for col in df.columns:
    if any(keyword in col.lower() for keyword in text_keywords):
        text_col = col
        break

print(f"\nText column identified: '{text_col}'")

# Find category column
category_keywords = ['product', 'category', 'type', 'issue', 'class']
category_col = None

for col in df.columns:
    if any(keyword in col.lower() for keyword in category_keywords):
        category_col = col
        break

print(f"Category column identified: '{category_col}'")

# Verify we found them
if text_col is None or category_col is None:
    print("\n ERROR: Could not identify columns!")
    print("Please manually specify:")
    print("  text_col = 'YOUR_TEXT_COLUMN_NAME'")
    print("  category_col = 'YOUR_CATEGORY_COLUMN_NAME'")
    exit()

STEP 3: Identifying text and category columns...

Text column identified: 'Issue'
Category column identified: 'Product'


In [19]:

print("STEP 4: Extracting needed columns...")


print(f"\nExtracting: '{text_col}' and '{category_col}'")

df_clean = df[[text_col, category_col]].copy()

print(f"Before cleaning: {len(df_clean)} rows")

STEP 4: Extracting needed columns...

Extracting: 'Issue' and 'Product'
Before cleaning: 1282355 rows


In [20]:

print("STEP 5: Handling missing values...")


print(f"\nMissing values in {text_col}: {df_clean[text_col].isnull().sum()}")
print(f"Missing values in {category_col}: {df_clean[category_col].isnull().sum()}")

# Remove rows with NaN
df_clean = df_clean.dropna(subset=[text_col, category_col])

print(f"After removing NaN: {len(df_clean)} rows")

STEP 5: Handling missing values...

Missing values in Issue: 0
Missing values in Product: 0
After removing NaN: 1282355 rows


In [21]:

print("STEP 6: Cleaning text data...")


# Convert to string
df_clean[text_col] = df_clean[text_col].astype(str)

# Remove extra whitespace
df_clean[text_col] = df_clean[text_col].str.strip()

# Remove rows with empty text
df_clean = df_clean[df_clean[text_col].str.len() > 0]

print(f"After removing empty text: {len(df_clean)} rows")

STEP 6: Cleaning text data...
After removing empty text: 1282355 rows


In [22]:

print("STEP 7: Handling duplicates...")


print(f"\nDuplicate text entries: {df_clean[text_col].duplicated().sum()}")

# Keep first occurrence of duplicates
df_clean = df_clean.drop_duplicates(subset=[text_col], keep='first')

print(f"After removing duplicates: {len(df_clean)} rows")

STEP 7: Handling duplicates...

Duplicate text entries: 1282188
After removing duplicates: 167 rows


In [23]:
print("STEP 8: Analyzing categories...")

print(f"\nTotal unique categories: {df_clean[category_col].nunique()}")
print(f"\nCategory distribution:")
print(df_clean[category_col].value_counts())

# Keep only categories with enough samples (optional)
min_samples = 50
category_counts = df_clean[category_col].value_counts()
valid_categories = category_counts[category_counts >= min_samples].index

print(f"\nKeeping categories with >= {min_samples} samples")
print(f"Before filtering: {len(df_clean)} rows")

df_clean = df_clean[df_clean[category_col].isin(valid_categories)]

print(f"After filtering: {len(df_clean)} rows")
print(f"Remaining categories: {df_clean[category_col].nunique()}")

STEP 8: Analyzing categories...

Total unique categories: 17

Category distribution:
Product
Credit card                                                                     32
Credit card or prepaid card                                                     18
Payday loan, title loan, or personal loan                                       18
Money transfer, virtual currency, or money service                              14
Consumer Loan                                                                   11
Mortgage                                                                        11
Credit reporting, credit repair services, or other personal consumer reports    10
Debt collection                                                                 10
Student loan                                                                     6
Payday loan                                                                      6
Prepaid card                                                                 

In [24]:

print("STEP 9: Renaming columns...")

df_clean.columns = ["Ticket Description", "Ticket Type"]

print("Columns renamed to:")
print(f"  - Ticket Description")
print(f"  - Ticket Type")

STEP 9: Renaming columns...
Columns renamed to:
  - Ticket Description
  - Ticket Type


In [25]:

print("STEP 10: Final verification...")


print(f"\nFinal dataset shape: {df_clean.shape}")
print(f"Total records: {len(df_clean)}")
print(f"Total categories: {df_clean['Ticket Type'].nunique()}")

print("\nFinal category distribution:")
print(df_clean['Ticket Type'].value_counts())

print("\nSample data:")
print(df_clean.head(3))

STEP 10: Final verification...

Final dataset shape: (0, 2)
Total records: 0
Total categories: 0

Final category distribution:
Series([], Name: count, dtype: int64)

Sample data:
Empty DataFrame
Columns: [Ticket Description, Ticket Type]
Index: []


In [26]:

print("STEP 11: Saving cleaned data...")

# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

output_path = 'data/complaints_cleaned.csv'

df_clean.to_csv(output_path, index=False)

print(f"✓ Cleaned data saved to: {output_path}")

STEP 11: Saving cleaned data...
✓ Cleaned data saved to: data/complaints_cleaned.csv


In [27]:

print("PREPROCESSING COMPLETE!")

print(f"""
Summary:
--------
✓ Loaded: {csv_path}
✓ Records processed: {len(df_clean)}
✓ Categories: {df_clean['Ticket Type'].nunique()}
✓ Output file: {output_path}

Ready for training!
""")


PREPROCESSING COMPLETE!

Summary:
--------
✓ Loaded: /Users/sristiprasad/Documents/GitHub/customer_complaint_classifier/data/raw_data/consumer_complaints.csv
✓ Records processed: 0
✓ Categories: 0
✓ Output file: data/complaints_cleaned.csv

Ready for training!



In [28]:
import os

# Save cleaned data
os.makedirs('data/preprocess_data', exist_ok=True)
df_clean.to_csv('data/preprocess_data/complaints_cleaned.csv', index=False)

print("✓ Cleaned data saved to: data/preprocess_data/complaints_cleaned.csv")
print(f"✓ Total records: {len(df_clean):,}")
print(f"✓ Categories: {df_clean['Ticket Type'].nunique()}")

✓ Cleaned data saved to: data/preprocess_data/complaints_cleaned.csv
✓ Total records: 0
✓ Categories: 0
